# 04 — Rule Engine Pattern

Purpose: define validation rules as data/config, then apply them in one reusable function.

Process schema:

```text
BRONZE INPUT
      |
      v
RULES LIST
- missing_user_id
- invalid_event_type
- missing_amount
- negative_amount
- invalid_event_time
      |
      v
APPLY RULE ENGINE
      |
      +--> SILVER
      |
      +--> QUARANTINE
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("rule-engine-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Bronze input

In [2]:
bronze_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time_raw", StringType(), True),
])

rows = [
    ("e1", "u1", "purchase", 10.0, "2026-01-01 10:00:00"),
    ("e2", None, "purchase", 20.0, "2026-01-01 10:05:00"),
    ("e3", "u2", "refund", 5.0, "2026-01-01 10:10:00"),
    ("e4", "u3", "unknown", 7.0, "2026-01-01 10:15:00"),
    ("e5", "u4", "purchase", -1.0, "2026-01-01 10:20:00"),
    ("e6", "u5", "refund", None, "2026-01-01 10:25:00"),
    ("e7", "u6", "purchase", 12.0, "bad-date"),
]

bronze = spark.createDataFrame(rows, bronze_schema)

bronze.show(truncate=False)

+--------+-------+----------+------+-------------------+
|event_id|user_id|event_type|amount|event_time_raw     |
+--------+-------+----------+------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|
|e2      |NULL   |purchase  |20.0  |2026-01-01 10:05:00|
|e3      |u2     |refund    |5.0   |2026-01-01 10:10:00|
|e4      |u3     |unknown   |7.0   |2026-01-01 10:15:00|
|e5      |u4     |purchase  |-1.0  |2026-01-01 10:20:00|
|e6      |u5     |refund    |NULL  |2026-01-01 10:25:00|
|e7      |u6     |purchase  |12.0  |bad-date           |
+--------+-------+----------+------+-------------------+



## Step 2 — Normalize and parse

In [3]:
prepared = (
    bronze
    .withColumn("event_type", F.lower(F.trim(F.col("event_type"))))
    .withColumn("user_id", F.trim(F.col("user_id")))
    .withColumn("event_time", F.to_timestamp("event_time_raw"))
)

prepared.show(truncate=False)

[Stage 2:>                                                          (0 + 1) / 1]

+--------+-------+----------+------+-------------------+-------------------+
|event_id|user_id|event_type|amount|event_time_raw     |event_time         |
+--------+-------+----------+------+-------------------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01 10:00:00|
|e2      |NULL   |purchase  |20.0  |2026-01-01 10:05:00|2026-01-01 10:05:00|
|e3      |u2     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01 10:10:00|
|e4      |u3     |unknown   |7.0   |2026-01-01 10:15:00|2026-01-01 10:15:00|
|e5      |u4     |purchase  |-1.0  |2026-01-01 10:20:00|2026-01-01 10:20:00|
|e6      |u5     |refund    |NULL  |2026-01-01 10:25:00|2026-01-01 10:25:00|
|e7      |u6     |purchase  |12.0  |bad-date           |NULL               |
+--------+-------+----------+------+-------------------+-------------------+



## Step 3 — Rules list

Each rule has:

```text
rule_name
condition
```

The first matching rule becomes `error_reason`.

In [4]:
allowed_event_types = ["purchase", "refund"]

rules = [
    ("missing_event_id", F.col("event_id").isNull()),
    ("missing_user_id", F.col("user_id").isNull() | (F.col("user_id") == "")),
    ("invalid_event_type", ~F.col("event_type").isin(allowed_event_types)),
    ("missing_amount", F.col("amount").isNull()),
    ("negative_amount", F.col("amount") < 0),
    ("invalid_event_time", F.col("event_time").isNull()),
]

## Step 4 — Apply rule engine

This function builds one `error_reason` column from the rules list.

In [5]:
def apply_rule_engine(df, rules):
    error_expr = None

    for rule_name, condition in rules:
        if error_expr is None:
            error_expr = F.when(condition, F.lit(rule_name))
        else:
            error_expr = error_expr.when(condition, F.lit(rule_name))

    return (
        df
        .withColumn("error_reason", error_expr)
        .withColumn("is_valid", F.col("error_reason").isNull())
    )

## Step 5 — Run validation

In [6]:
validated = apply_rule_engine(prepared, rules)

validated.select(
    "event_id",
    "user_id",
    "event_type",
    "amount",
    "event_time_raw",
    "event_time",
    "is_valid",
    "error_reason",
).show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+--------+------------------+
|event_id|user_id|event_type|amount|event_time_raw     |event_time         |is_valid|error_reason      |
+--------+-------+----------+------+-------------------+-------------------+--------+------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01 10:00:00|true    |NULL              |
|e2      |NULL   |purchase  |20.0  |2026-01-01 10:05:00|2026-01-01 10:05:00|false   |missing_user_id   |
|e3      |u2     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01 10:10:00|true    |NULL              |
|e4      |u3     |unknown   |7.0   |2026-01-01 10:15:00|2026-01-01 10:15:00|false   |invalid_event_type|
|e5      |u4     |purchase  |-1.0  |2026-01-01 10:20:00|2026-01-01 10:20:00|false   |negative_amount   |
|e6      |u5     |refund    |NULL  |2026-01-01 10:25:00|2026-01-01 10:25:00|false   |missing_amount    |
|e7      |u6     |purchase  |12.0  |bad-date           

## Step 6 — Silver output

In [7]:
silver = (
    validated
    .filter(F.col("is_valid"))
    .select(
        "event_id",
        "user_id",
        "event_type",
        "amount",
        "event_time",
    )
)

silver.show(truncate=False)

+--------+-------+----------+------+-------------------+
|event_id|user_id|event_type|amount|event_time         |
+--------+-------+----------+------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|
|e3      |u2     |refund    |5.0   |2026-01-01 10:10:00|
+--------+-------+----------+------+-------------------+



## Step 7 — Quarantine output

In [8]:
quarantine = (
    validated
    .filter(~F.col("is_valid"))
    .select(
        "event_id",
        "user_id",
        "event_type",
        "amount",
        "event_time_raw",
        "error_reason",
    )
)

quarantine.show(truncate=False)

+--------+-------+----------+------+-------------------+------------------+
|event_id|user_id|event_type|amount|event_time_raw     |error_reason      |
+--------+-------+----------+------+-------------------+------------------+
|e2      |NULL   |purchase  |20.0  |2026-01-01 10:05:00|missing_user_id   |
|e4      |u3     |unknown   |7.0   |2026-01-01 10:15:00|invalid_event_type|
|e5      |u4     |purchase  |-1.0  |2026-01-01 10:20:00|negative_amount   |
|e6      |u5     |refund    |NULL  |2026-01-01 10:25:00|missing_amount    |
|e7      |u6     |purchase  |12.0  |bad-date           |invalid_event_time|
+--------+-------+----------+------+-------------------+------------------+



## Step 8 — Add another rule

Example: maximum amount allowed.

In [9]:
rules_with_max_amount = rules + [
    ("amount_too_large", F.col("amount") > 1000.0),
]

extra_rows = [
    ("e8", "u7", "purchase", 5000.0, "2026-01-01 11:00:00"),
]

extra_df = spark.createDataFrame(extra_rows, bronze_schema)

extra_prepared = (
    extra_df
    .withColumn("event_type", F.lower(F.trim(F.col("event_type"))))
    .withColumn("user_id", F.trim(F.col("user_id")))
    .withColumn("event_time", F.to_timestamp("event_time_raw"))
)

validated_extra = apply_rule_engine(extra_prepared, rules_with_max_amount)

validated_extra.select(
    "event_id",
    "user_id",
    "event_type",
    "amount",
    "is_valid",
    "error_reason",
).show(truncate=False)

+--------+-------+----------+------+--------+----------------+
|event_id|user_id|event_type|amount|is_valid|error_reason    |
+--------+-------+----------+------+--------+----------------+
|e8      |u7     |purchase  |5000.0|false   |amount_too_large|
+--------+-------+----------+------+--------+----------------+



## Step 9 — Control totals

In [10]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("bronze_rows", bronze.count()),
        ("validated_rows", validated.count()),
        ("silver_rows", silver.count()),
        ("quarantine_rows", quarantine.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+---------------+-----+
|metric         |value|
+---------------+-----+
|bronze_rows    |7    |
|validated_rows |7    |
|silver_rows    |2    |
|quarantine_rows|5    |
+---------------+-----+



## Final result

```text
BRONZE
  |
  v
PREPARE COLUMNS
  |
  v
APPLY RULE ENGINE
  |
  +--> SILVER
  |
  +--> QUARANTINE
```

In [11]:
spark.stop()